# Data Cleaning & Visualization Project
This notebook uses a **synthetic sample retail sales dataset** for demonstration.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'dataset' / 'raw_sales_data.csv'
OUTPUT_DIR = BASE_DIR / 'outputs'
CHART_DIR = BASE_DIR / 'visualizations'
OUTPUT_DIR.mkdir(exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH)
df.head()


## 1. Inspect data quality


In [ ]:
print('Shape:', df.shape)
display(df.isna().sum().to_frame('Missing values'))
print('Duplicate rows:', df.duplicated().sum())
df.describe(include='all')


## 2. Clean missing values and duplicates


In [ ]:
df = df.drop_duplicates().copy()
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
for col in ['Quantity', 'Unit_Price', 'Sales']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Product'] = df['Product'].fillna('Unknown')
df['Region'] = df['Region'].fillna('Unknown')
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Unit_Price'] = df['Unit_Price'].fillna(df['Unit_Price'].median())
df['Sales'] = df['Quantity'] * df['Unit_Price']
df.isna().sum()


## 3. Detect and cap outliers with the IQR rule


In [ ]:
q1, q3 = df['Sales'].quantile([0.25, 0.75])
iqr = q3 - q1
lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr
outlier_count = ((df['Sales'] < lower) | (df['Sales'] > upper)).sum()
print('Outliers detected:', int(outlier_count))
df['Sales'] = df['Sales'].clip(lower, upper)
df.head()


## 4. Save cleaned data and summarize sales


In [ ]:
df.to_csv(OUTPUT_DIR / 'cleaned_sales_data.csv', index=False)
summary = df.groupby('Product')['Sales'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)
summary.to_csv(OUTPUT_DIR / 'product_sales_summary.csv')
display(summary)


## 5. Visualize findings


In [ ]:
product_sales = df.groupby('Product')['Sales'].sum().sort_values(ascending=False)
product_sales.plot(kind='bar', title='Total Sales by Product', xlabel='Product', ylabel='Sales', rot=30)
plt.tight_layout(); plt.show()


In [ ]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
region_sales.plot(kind='bar', title='Total Sales by Region', xlabel='Region', ylabel='Sales')
plt.tight_layout(); plt.show()


In [ ]:
daily_sales = df.groupby('Order_Date')['Sales'].sum().sort_index()
daily_sales.plot(kind='line', marker='o', title='Daily Sales Trend', xlabel='Order Date', ylabel='Sales', figsize=(10,5))
plt.tight_layout(); plt.show()


## Conclusion
The workflow demonstrates duplicate removal, missing-value handling, outlier treatment, and visual reporting. Because the data is synthetic, describe results as sample findings rather than real business conclusions.
